In [ ]:
!hf auth login  # unnecessary


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? [y/N]: N
Token is valid (permission: write).
The token `568_whisper` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `568_whisper

In [1]:
!pip install faster-whisper
!pip install ffmpeg-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 63.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 90.3 MB/s eta 0:00:00


In [4]:
from IPython.display import Javascript, display
from google.colab import output
from base64 import b64decode

record_js = """
async function recordAudio() {
  // Create a stop button
  const div = document.createElement('div');
  const stopButton = document.createElement('button');
  stopButton.textContent = '⏹️ Stop Recording';
  stopButton.style.padding = '10px 20px';
  stopButton.style.fontSize = '16px';
  stopButton.style.backgroundColor = '#ff4c4c';
  stopButton.style.color = 'white';
  stopButton.style.border = 'none';
  stopButton.style.borderRadius = '5px';
  stopButton.style.cursor = 'pointer';
  stopButton.style.marginTop = '10px';

  div.appendChild(stopButton);
  document.body.appendChild(div);

  const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
  const recorder = new MediaRecorder(stream);
  let chunks = [];

  recorder.ondataavailable = e => chunks.push(e.data);
  recorder.start();

  // Wait for button click instead of timeout
  await new Promise(resolve => {
    stopButton.onclick = resolve;
  });

  recorder.stop();
  stopButton.disabled = true;
  stopButton.textContent = '⏳ Processing...';
  stopButton.style.backgroundColor = '#cccccc';

  await new Promise(resolve => recorder.onstop = resolve);

  const blob = new Blob(chunks, { type: 'audio/webm' });
  const arrayBuffer = await blob.arrayBuffer();
  const bytes = new Uint8Array(arrayBuffer);

  let binary = '';
  for (let i = 0; i < bytes.byteLength; i++) {
    binary += String.fromCharCode(bytes[i]);
  }

  stream.getTracks().forEach(track => track.stop());
  div.remove(); // Clean up the button
  return btoa(binary);
}
"""

display(Javascript(record_js))
print("Recording started. Click the 'Stop Recording' button to stop...")

audio_base64 = output.eval_js("recordAudio()")
audio_bytes = b64decode(audio_base64)

with open("recorded_audio.webm", "wb") as f:
    f.write(audio_bytes)

print("Saved recorded_audio.webm")

<IPython.core.display.Javascript object>

Recording started. Click the 'Stop Recording' button to stop...
Saved recorded_audio.webm


In [5]:
import ffmpeg

(
    ffmpeg
    .input("recorded_audio.webm")
    .output("audio.wav", ac=1, ar=16000)
    .overwrite_output()
    .run(quiet=True)
)

(b'',
 b"ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers\n  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)\n  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-li

In [ ]:
# from faster_whisper import WhisperModel

# model_size = "large-v3"

# # GPU FP16
# model = WhisperModel(model_size, device="cuda", compute_type="float16")

# segments, info = model.transcribe("audio.wav", beam_size=3, temperature=0.3)

# print("Detected language '%s' with probability %f" % (info.language, info.language_probability))

# full_text = []
# for segment in segments:
#     print("[%.2fs -> %.2fs] %s" % (segment.start, segment.end, segment.text))
#     full_text.append(segment.text)

# final_text = " ".join(full_text)
# print("\nFinal transcription:")
# print(final_text)

Detected language 'en' with probability 0.794922

Final transcription:



In [ ]:
# with open("transcription.txt", "w", encoding="utf-8") as f:
#     f.write(final_text)

# print("Saved transcription to transcription.txt")

In [6]:
from faster_whisper import WhisperModel
from dataclasses import asdict

model_size = "large-v3"
model = WhisperModel(model_size, device="cuda", compute_type="float16")

segments, info = model.transcribe(
    "audio.wav",
    beam_size=3,
    temperature=0.3,
    word_timestamps=True,
    vad_filter=True,   # optional, but often helpful
)

segments = list(segments)  # materialize generator once

print(f"Detected language '{info.language}' with probability {info.language_probability:.3f}")

for seg in segments:
    print(f"[{seg.start:.2f}s -> {seg.end:.2f}s] {seg.text}")
    if seg.words:
        for w in seg.words:
            print(f"   {w.start:.2f}-{w.end:.2f}  {w.word}  prob={w.probability:.3f}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Detected language 'en' with probability 0.948
[0.00s -> 4.00s]  Effective workplace communication is essential for success,
   0.00-0.74   Effective  prob=0.779
   0.74-1.06   workplace  prob=0.972
   1.06-1.74   communication  prob=0.985
   1.74-2.46   is  prob=0.995
   2.46-2.94   essential  prob=0.994
   2.94-3.40   for  prob=0.986
   3.40-4.00   success,  prob=0.989
[4.26s -> 7.90s]  yet many managers lack access to tailored and sustained training.
   4.26-4.42   yet  prob=0.833
   4.42-4.78   many  prob=0.986
   4.78-5.22   managers  prob=0.989
   5.22-5.56   lack  prob=0.947
   5.56-6.02   access  prob=0.987
   6.02-6.28   to  prob=0.998
   6.28-6.66   tailored  prob=0.224
   6.66-6.94   and  prob=0.987
   6.94-7.28   sustained  prob=0.954
   7.28-7.90   training.  prob=0.958


In [14]:
all_words = []
for seg in segments:
    if seg.words:
        for w in seg.words:
            all_words.append({
                "word": w.word.strip(),
                "start": w.start,
                "end": w.end,
                "prob": w.probability,
                "segment_avg_logprob": seg.avg_logprob,
                "segment_no_speech_prob": seg.no_speech_prob,
            })

full_text = " ".join(w["word"] for w in all_words).strip()
print("\nFinal transcription:")
print(full_text)


Final transcription:
Effective workplace communication is essential for success, yet many managers lack access to tailored and sustained training.


In [15]:
def compute_speech_rate(words):
    if len(words) < 2:
        return {
            "num_words": len(words),
            "duration_sec": 0.0,
            "words_per_min_total": 0.0,
            "words_per_min_speaking_only": 0.0,
        }

    num_words = len(words)
    total_duration = words[-1]["end"] - words[0]["start"]

    speaking_time = sum(max(0.0, w["end"] - w["start"]) for w in words)

    wpm_total = (num_words / total_duration) * 60 if total_duration > 0 else 0.0
    wpm_speaking = (num_words / speaking_time) * 60 if speaking_time > 0 else 0.0

    return {
        "num_words": num_words,
        "duration_sec": round(total_duration, 2),
        "words_per_min_total": round(wpm_total, 2),
        "words_per_min_speaking_only": round(wpm_speaking, 2),
    }

In [16]:
def compute_pauses(words, pause_threshold=0.5):
    pauses = []

    for i in range(len(words) - 1):
        gap = words[i + 1]["start"] - words[i]["end"]
        if gap >= pause_threshold:
            pauses.append({
                "after_word": words[i]["word"],
                "before_word": words[i + 1]["word"],
                "pause_sec": round(gap, 3),
                "start": words[i]["end"],
                "end": words[i + 1]["start"],
            })

    return {
        "pause_count": len(pauses),
        "avg_pause_sec": round(sum(p["pause_sec"] for p in pauses) / len(pauses), 3) if pauses else 0.0,
        "max_pause_sec": round(max((p["pause_sec"] for p in pauses), default=0.0), 3),
        "pauses": pauses,
    }

In [17]:
import re

FILLERS = {
    "um", "uh", "erm", "ah", "like", "you know", "i mean", "sort of", "kind of"
}

def compute_filler_words(text):
    text_l = text.lower()

    # multi-word fillers first
    multi_word_fillers = ["you know", "i mean", "sort of", "kind of"]
    multi_counts = {f: len(re.findall(rf"\b{re.escape(f)}\b", text_l)) for f in multi_word_fillers}

    # tokenize simple single words
    tokens = re.findall(r"\b[\w']+\b", text_l)
    single_fillers = {"um", "uh", "erm", "ah", "like"}
    single_counts = {f: tokens.count(f) for f in single_fillers}

    counts = {**multi_counts, **single_counts}
    total = sum(counts.values())

    return {
        "filler_total": total,
        "filler_breakdown": counts
    }

In [18]:
def find_low_confidence_words(words, threshold=0.60):
    low = [w for w in words if w["prob"] < threshold and w["word"]]
    return {
        "count": len(low),
        "words": low
    }

In [19]:
def merge_low_confidence_phrases(low_words, max_gap=0.25):
    if not low_words:
        return []

    low_words = sorted(low_words, key=lambda x: x["start"])
    phrases = []
    current = [low_words[0]]

    for w in low_words[1:]:
        if w["start"] - current[-1]["end"] <= max_gap:
            current.append(w)
        else:
            phrases.append(current)
            current = [w]
    phrases.append(current)

    merged = []
    for phrase in phrases:
        merged.append({
            "text": " ".join(w["word"] for w in phrase),
            "start": phrase[0]["start"],
            "end": phrase[-1]["end"],
            "avg_prob": round(sum(w["prob"] for w in phrase) / len(phrase), 3)
        })
    return merged

In [20]:
def detect_repetitions(words):
    reps = []
    for i in range(len(words) - 1):
        w1 = words[i]["word"].lower()
        w2 = words[i + 1]["word"].lower()
        if w1 and w1 == w2:
            reps.append({
                "word": w1,
                "start": words[i]["start"],
                "end": words[i + 1]["end"]
            })
    return {
        "repetition_count": len(reps),
        "repetitions": reps
    }

In [21]:
speech_rate = compute_speech_rate(all_words)
pause_stats = compute_pauses(all_words, pause_threshold=0.5)
filler_stats = compute_filler_words(full_text)
low_conf_stats = find_low_confidence_words(all_words, threshold=0.60)
low_conf_phrases = merge_low_confidence_phrases(low_conf_stats["words"], max_gap=0.25)
repetition_stats = detect_repetitions(all_words)

analysis = {
    "speech_rate": speech_rate,
    "pause_stats": pause_stats,
    "filler_stats": filler_stats,
    "low_confidence": {
        "word_count": low_conf_stats["count"],
        "phrases": low_conf_phrases
    },
    "repetition_stats": repetition_stats,
}

from pprint import pprint
pprint(analysis)

{'filler_stats': {'filler_breakdown': {'ah': 0,
                                       'erm': 0,
                                       'i mean': 0,
                                       'kind of': 0,
                                       'like': 0,
                                       'sort of': 0,
                                       'uh': 0,
                                       'um': 0,
                                       'you know': 0},
                  'filler_total': 0},
 'low_confidence': {'phrases': [{'avg_prob': np.float64(0.224),
                                 'end': np.float64(6.66),
                                 'start': np.float64(6.28),
                                 'text': 'tailored'}],
                    'word_count': 1},
 'pause_stats': {'avg_pause_sec': 0.0,
                 'max_pause_sec': 0.0,
                 'pause_count': 0,
                 'pauses': []},
 'repetition_stats': {'repetition_count': 0, 'repetitions': []},
 'speech_rate': {'dur